# XGBoost: Learning from Mistakes

**Chapter 5: Advanced Classification Methods**

## Learning Objectives

- Understand gradient boosting and how it differs from bagging
- Build an XGBoost model for xG prediction
- Compare XGBoost with Random Forest
- Learn key XGBoost hyperparameters
- Understand when to use XGBoost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsbombpy import sb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"XGBoost version: {xgb.__version__}")

## The Intuition: Boosting vs Bagging

### Random Forest (Bagging)
- Trains trees **in parallel**
- Each tree is **independent**
- Trees vote with **equal weight**
- Reduces variance through **averaging**

### XGBoost (Boosting)
- Trains trees **sequentially**
- Each tree **learns from previous mistakes**
- Trees have **different weights**
- Reduces bias through **iterative correction**

**Analogy:** Random Forest is like asking 100 experts independently. XGBoost is like a student taking 100 practice tests, learning from each mistake.

## Load Data: xG Prediction

We'll build an xG model using shot data.

In [ ]:
# Load shot data
matches = sb.matches(competition_id=16, season_id=4)  # Champions League
events = sb.events(match_id=22912)  # Liverpool vs Tottenham Final
shots = events[events['type'] == 'Shot'].copy()

print(f"Total shots: {len(shots)}")
print(f"\nShot outcomes:")
print(shots['shot_outcome'].value_counts())

## Feature Engineering for xG

In [ ]:
# Extract shot features
def extract_shot_features(shots_df):
    df = shots_df.copy()
    
    # Distance to goal (center of goal is at x=120, y=40)
    df['distance_to_goal'] = df['location'].apply(
        lambda loc: np.sqrt((120 - loc[0])**2 + (40 - loc[1])**2) if isinstance(loc, list) else None
    )
    
    # Angle to goal
    def calculate_angle(loc):
        if not isinstance(loc, list):
            return None
        x, y = loc[0], loc[1]
        angle = np.abs(np.arctan((40 - y) / (120 - x))) * 180 / np.pi
        return angle
    
    df['angle_to_goal'] = df['location'].apply(calculate_angle)
    
    # Body part (1 if right foot, 0 otherwise)
    df['right_foot'] = (df['shot_body_part'] == 'Right Foot').astype(int)
    
    # Shot type (1 if open play, 0 if set piece)
    df['open_play'] = (df['shot_type'] == 'Open Play').astype(int)
    
    # Target variable: 1 if goal, 0 otherwise
    df['goal'] = (df['shot_outcome'] == 'Goal').astype(int)
    
    return df

shots = extract_shot_features(shots)

print(f"\nFeature summary:")
print(shots[['distance_to_goal', 'angle_to_goal', 'right_foot', 'open_play', 'goal']].describe())
print(f"\nGoal rate: {shots['goal'].mean():.2%}")

## Prepare Data

In [ ]:
# Select features
feature_cols = ['distance_to_goal', 'angle_to_goal', 'right_foot', 'open_play']
X = shots[feature_cols].dropna()
y = shots.loc[X.index, 'goal']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features (XGBoost doesn't require scaling, but it can help)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {len(X_train)} shots")
print(f"Test set: {len(X_test)} shots")
print(f"\nClass balance:")
print(f"Goals: {y_train.sum()} ({y_train.mean():.1%})")
print(f"Non-goals: {(~y_train.astype(bool)).sum()} ({(1-y_train.mean()):.1%})")

## Train XGBoost Model

In [ ]:
# Create XGBoost classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=100,      # Number of boosting rounds
    max_depth=3,           # Maximum tree depth
    learning_rate=0.1,     # Step size shrinkage
    objective='binary:logistic',  # Binary classification
    eval_metric='logloss', # Evaluation metric
    random_state=42
)

# Train model
xgb_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_pred_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
logloss_xgb = log_loss(y_test, y_pred_proba_xgb)

print(f"XGBoost Performance:")
print(f"Accuracy: {accuracy_xgb:.3f}")
print(f"AUC-ROC: {auc_xgb:.3f}")
print(f"Log Loss: {logloss_xgb:.3f}")

## Compare with Random Forest

In [ ]:
# Train Random Forest for comparison
rf_model = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

accuracy_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
logloss_rf = log_loss(y_test, y_pred_proba_rf)

print(f"Random Forest Performance:")
print(f"Accuracy: {accuracy_rf:.3f}")
print(f"AUC-ROC: {auc_rf:.3f}")
print(f"Log Loss: {logloss_rf:.3f}")

print(f"\n--- Comparison ---")
print(f"XGBoost AUC: {auc_xgb:.3f}")
print(f"Random Forest AUC: {auc_rf:.3f}")
print(f"Improvement: {(auc_xgb - auc_rf)*100:.1f} percentage points")

## Feature Importance in XGBoost

In [ ]:
# Get feature importances
importances = xgb_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("Feature Importances:")
print(feature_importance_df.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
plt.xlabel('Importance', fontsize=12)
plt.title('XGBoost Feature Importance for xG Prediction', fontsize=14)
plt.tight_layout()
plt.show()

## Understanding XGBoost Parameters

### Learning Parameters
- **learning_rate** (eta): Controls contribution of each tree (0.01-0.3)
  - Lower = more robust but slower
- **n_estimators**: Number of boosting rounds (50-1000)
  - More trees = better performance (with diminishing returns)

### Tree Parameters
- **max_depth**: Maximum depth of trees (3-10)
  - Deeper = more complex patterns
- **min_child_weight**: Minimum sum of instance weight in a child (1-10)
  - Higher = more conservative

### Regularization
- **reg_alpha**: L1 regularization (0-1)
- **reg_lambda**: L2 regularization (0-1)
  - Both prevent overfitting

## XGBoost vs Random Forest: When to Use Which

| Criterion | Random Forest | XGBoost |
|-----------|---------------|----------|
| **Accuracy** | Good | Excellent |
| **Training Speed** | Fast | Slow |
| **Interpretability** | Good (feature importance) | Moderate (needs SHAP) |
| **Hyperparameter Tuning** | Minimal | Extensive |
| **Robustness** | Excellent | Good |
| **Best For** | Quick baseline, small data | Maximum accuracy, competitions |

## Summary

In this notebook, we:

1. ✅ Understood boosting (sequential learning from mistakes)
2. ✅ Built an XGBoost model for xG prediction
3. ✅ Compared XGBoost with Random Forest
4. ✅ Learned key XGBoost hyperparameters
5. ✅ Understood when to use each algorithm

## Key Takeaways

- XGBoost = **sequential trees learning from errors**
- **More accurate** than Random Forest (typically 2-5% improvement)
- Requires **more tuning** and computational resources
- Best for **production systems** where accuracy is critical
- Random Forest is often **good enough** for exploratory work

## Next Steps

XGBoost is powerful but complex. How do we interpret its predictions? **SHAP analysis** provides the answer!

## Practice Exercises

1. **Add More Features**: Include shot technique, pressure, game state
2. **Tune Learning Rate**: Try values from 0.01 to 0.3
3. **Early Stopping**: Use eval_set to stop training when performance plateaus
4. **Class Weights**: Use scale_pos_weight to handle class imbalance
5. **Cross-Validation**: Use xgb.cv() for robust evaluation